In [1]:
!git clone https://github.com/Uness10/HopefullyItWorks.git

Cloning into 'HopefullyItWorks'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 24 (delta 6), reused 23 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 33.32 KiB | 6.66 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [9]:
%cd HopefullyItWorks

/content/HopefullyItWorks


In [3]:
import os

# Check if the outputs (1).zip file exists
if os.path.exists('outputs (1).zip'):
  !unzip -q 'outputs (1).zip'
  print('outputs (1).zip unzipped successfully.')
elif os.path.exists('outputs.zip'):
  !unzip -q outputs.zip
  print('outputs.zip unzipped successfully.')
else:
  print('Neither outputs.zip nor outputs (1).zip found in the current directory.')

# Verify content
!ls -F

outputs (1).zip unzipped successfully.
 HopefullyItWorks/   outputs/  'outputs (1).zip'   sample_data/


In [4]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load base model + LoRA adapter
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_path = "outputs/qwen_agri_lora"  # your --output-dir

tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

def ask(question: str, system_prompt: str = "You are an expert agriculture assistant. Give practical, concise, field-ready advice."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
        )

    # Decode only the new tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# Test it
print(ask("What is the best fertilizer for wheat crops?"))
print(ask("How do I treat fungal infections in tomatoes?"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

For wheat crops, it is important to balance nitrogen intake with micronutrients and other essential elements to ensure optimal plant growth and yield. A balanced fertilizer that provides a mix of nitrogen, phosphorus, and potassium is generally recommended, as these elements play crucial roles in plant development and overall health.
Use disease-free seeds and plants. Plant during periods of low rainfall or poor growth conditions to reduce the risk of tomato blight. Thoroughly clean and disinfect planting tools and equipment before handling tomato crops to prevent the spread of bacterial blight. Use a seed drill to plant tomato seedlings at depths of 10-12 cm to avoid direct contact with infected plants. Avoid using bare-root seedlings as they can act as a reservoir for Bacterial Blight. Use a fungicide spray at 0.2% copper oxybendazim or 0.3% mancozeb at intervals of 7-10 days after planting to prevent infection by Bacterial Blight.


In [7]:
ask("How much phosphorus per hectare should I apply to wheat?")


'The recommended amount of phosphorus for wheat can vary depending on various factors such as the location and climate, soil fertility, and crop variety. However, as a general guideline, farmers typically apply 0.2-0.4% of total fertilizer application to wheat, which corresponds to about 10-15 kg/ha of phosphorus.'

In [11]:
! mkdir -p ./data/captions ./data/vqa ./checkpoints


In [14]:
! git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git /content/HopefullyItWorks/data/PlantVillage-Dataset

Cloning into '/content/HopefullyItWorks/data/PlantVillage-Dataset'...
remote: Enumerating objects: 163219, done.
remote: Counting objects: 100% (163219/163219), done.
remote: Compressing objects: 100% (163125/163125), done.
remote: Total 163219 (delta 93), reused 163214 (delta 93), pack-reused 0 (from 0)
Receiving objects: 100% (163219/163219), 2.00 GiB | 36.16 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Updating files: 100% (182404/182404), done.


In [15]:
! python /content/HopefullyItWorks/build_stage1_captions.py --images_dir /content/HopefullyItWorks/data/PlantVillage-Dataset/raw/color --output_jsonl /content/data/captions/captions.jsonl --shuffle --max_images 1000

[done] Wrote 1000 caption samples to: /content/data/captions/captions.jsonl


In [22]:
! python /content/HopefullyItWorks/build_stage2_vqa.py --images_dir /content/HopefullyItWorks/data/PlantVillage-Dataset/raw/color --output_jsonl /content/HopefullyItWorks/data/vqa/vqa.jsonl --pairs_per_image 4 --shuffle --max_images 250

[done] Wrote 1000 VQA samples to: /content/HopefullyItWorks/data/vqa/vqa.jsonl
[note] This dataset is template-generated. Manually review a sample before training.


In [20]:
! python /content/HopefullyItWorks/train_projector.py --stage 1 --base_model Qwen/Qwen2.5-0.5B-Instruct --lora_ckpt /content/HopefullyItWorks/outputs/qwen_agri_lora --data_dir /content/data/captions --output_dir /content/HopefullyItWorks/checkpoints --epochs 3 --batch_size 4 --lr 1e-3

INFO:__main__:Stage 1 training on cuda
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-large-patch14/32bd64288804d66eefd0ccbe215aa642df71cc41/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-large-patch14/32bd64288804d66eefd0ccbe215aa642df71cc41/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100% 391/391 [00:00<00:00, 798.52it/s, Materializing param=vision_model.pre_layrnorm.weight]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14
Key       

In [28]:
! python /content/HopefullyItWorks/train_projector.py \
  --stage 2 \
  --base_model Qwen/Qwen2.5-0.5B-Instruct \
  --lora_ckpt /content/HopefullyItWorks/outputs/qwen_agri_lora \
  --projector_ckpt /content/HopefullyItWorks/checkpoints/projector_stage1.pt \
  --data_dir /content/HopefullyItWorks/data/vqa \
  --output_dir /content/HopefullyItWorks/checkpoints \
  --epochs 3 \
  --batch_size 4 \
  --lr 1e-3

INFO:__main__:Stage 2 training on cuda
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-large-patch14/32bd64288804d66eefd0ccbe215aa642df71cc41/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-large-patch14/32bd64288804d66eefd0ccbe215aa642df71cc41/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100% 391/391 [00:00<00:00, 871.67it/s, Materializing param=vision_model.pre_layrnorm.weight]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14
Key       

In [31]:
! python /content/HopefullyItWorks/inference.py \
  --image /content/HopefullyItWorks/data/PlantVillage-Dataset/raw/color/Apple___Apple_scab/"00075aa8-d81a-4184-8541-b692b78d398a___FREC_Scab 3335".JPG \
  --question "What disease is visible and how severe is it?" \
  --base_model Qwen/Qwen2.5-0.5B-Instruct \
  --agri_lora_ckpt /content/HopefullyItWorks/outputs/qwen_agri_lora \
  --visual_lora_ckpt /content/HopefullyItWorks/checkpoints/visual_lora_best \
  --projector_ckpt /content/HopefullyItWorks/checkpoints/projector_stage2_best.pt

[info] Using device: cuda
[info] Loading CLIP ViT...
Loading weights: 100% 391/391 [00:00<00:00, 837.85it/s, Materializing param=vision_model.pre_layrnorm.weight]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.w